In [1]:
import h5py
import numpy as np
import pandas as pd

# 打开 HDF5 文件（只读模式）
file_path = "/home/zme/data/robot_data/fold_clothes/episode_32.hdf5"  # 替换为你的 .h5 文件路径

with h5py.File(file_path, "r") as f:
    # 查看文件结构（顶级键）
    print("顶级组/数据集名称：")
    print(list(f.keys()))

    # 遍历所有数据集（递归查看结构可选）
    def print_structure(name, obj):
        print(f"{name}: {type(obj)}")
        if isinstance(obj, h5py.Dataset):
            print(f"  Shape: {obj.shape}, Dtype: {obj.dtype}")

    f.visititems(print_structure)

    # 假设有一个数据集叫 'data'
    if "action" in f:
        dataset = f["action"][:]  # 读取全部数据到内存（注意：大文件慎用）
        print("\n数据预览：")
        print(dataset[:5])  # 打印前5行

        # 转为 NumPy 数组（通常已经是）
        arr = np.array(dataset)

        # 简单分析（示例）
        print("\n基本统计信息：")
        print(f"均值: {arr.mean()}")
        print(f"标准差: {arr.std()}")
        print(f"最小值: {arr.min()}, 最大值: {arr.max()}")

        # 如果是表格型数据，可转为 pandas DataFrame
        # 假设每列有名字（或你知道列名）
        # df = pd.DataFrame(arr, columns=['col1', 'col2', 'col3'])
        # print(df.describe())


顶级组/数据集名称：
['action', 'observations']
action: <class 'h5py._hl.dataset.Dataset'>
  Shape: (705, 16), Dtype: float32
observations: <class 'h5py._hl.group.Group'>
observations/images: <class 'h5py._hl.group.Group'>
observations/images/left_wrist: <class 'h5py._hl.dataset.Dataset'>
  Shape: (705, 102824), Dtype: uint8
observations/images/right_wrist: <class 'h5py._hl.dataset.Dataset'>
  Shape: (705, 97025), Dtype: uint8
observations/images/top: <class 'h5py._hl.dataset.Dataset'>
  Shape: (705, 84931), Dtype: uint8
observations/qpos: <class 'h5py._hl.dataset.Dataset'>
  Shape: (705, 16), Dtype: float32
observations/qvel: <class 'h5py._hl.dataset.Dataset'>
  Shape: (705, 16), Dtype: float32

数据预览：
[[-2.129847    1.4816209   2.3936918  -1.9795852   0.81607777  0.00299978
  -1.5125732   1.         -1.9099083   1.4479415   2.273973   -1.9296796
   0.9923833   0.1686697  -1.8008252   1.        ]
 [-2.1314833   1.480121    2.3945098  -1.9842211   0.84362125  0.00613592
  -1.5148913   1.         

In [3]:
f = h5py.File(file_path, "r")

In [9]:
f["action"][:].shape

(705, 16)

In [12]:
f['observations']['qpos'][:, 13].shape

(705,)

In [13]:
abs(f['observations']['qpos'][:, 13] - f["action"][:][:, 13]) > 0.1

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False,

In [4]:
def count_leading_trailing_true(arr):
    arr = np.asarray(arr, dtype=bool)
    if arr.size == 0:
        return 0, 0

    # 计算开头连续 True 的数量
    if not arr[0]:
        leading = 0
    else:
        # 找到第一个 False 的位置
        first_false = np.argmax(~arr)
        leading = first_false if first_false > 0 else arr.size

    # 计算结尾连续 True 的数量
    if not arr[-1]:
        trailing = 0
    else:
        # 从末尾开始找第一个 False 的位置
        last_false = np.argmax(~arr[::-1])
        trailing = last_false if last_false > 0 else arr.size

    return leading, trailing

In [9]:
from pathlib import Path
import h5py
import numpy as np

h5_dataset = Path("/home/zme/data/robot_data/fold_clothes")
for file in h5_dataset.glob("*.hdf5"):
    with h5py.File(file, "r+") as f:
        diffs = abs(f['observations']['qpos'][:, 13] - f["action"][:][:, 13]) > 0.1
        diff2 = abs(f['observations']['qpos'][:, 1] - f["action"][:][:, 1]) > 0.01
        leading, trailing = count_leading_trailing_true(diffs)
        leading2, trailing2 = count_leading_trailing_true(diff2)
        leading = max(leading, leading2)
        trailing =  max(trailing, trailing2)

        action = f['action'][:]
        new_action = action[leading: len(action) - trailing]
        del f['action']
        f.create_dataset("action", data=new_action)

        qpos = f['observations']['qpos'][:]
        new_qpos = qpos[leading: len(qpos) - trailing]
        del f['observations']['qpos']
        f['observations'].create_dataset("qpos", data=new_qpos)

        obs_img_left = f['observations']['images']['left_wrist'][:]
        new_obs_img_left = obs_img_left[leading: len(obs_img_left) - trailing]
        del f['observations']['images']['left_wrist']
        f['observations']['images'].create_dataset("left_wrist", data=new_obs_img_left)

        obs_img_right = f['observations']['images']['right_wrist'][:]
        new_obs_img_right = obs_img_right[leading: len(obs_img_right) - trailing]
        del f['observations']['images']['right_wrist']
        f['observations']['images'].create_dataset("right_wrist", data=new_obs_img_right)

        obs_img_top = f['observations']['images']['top'][:]
        new_obs_img_top = obs_img_top[leading: len(obs_img_top) - trailing]
        del f['observations']['images']['top']
        f['observations']['images'].create_dataset("top", data=new_obs_img_top)
    print(file)

/home/zme/data/robot_data/fold_clothes/episode_20.hdf5
/home/zme/data/robot_data/fold_clothes/episode_80.hdf5
/home/zme/data/robot_data/fold_clothes/episode_55.hdf5
/home/zme/data/robot_data/fold_clothes/episode_8.hdf5
/home/zme/data/robot_data/fold_clothes/episode_152.hdf5
/home/zme/data/robot_data/fold_clothes/episode_50.hdf5
/home/zme/data/robot_data/fold_clothes/episode_125.hdf5
/home/zme/data/robot_data/fold_clothes/episode_48.hdf5
/home/zme/data/robot_data/fold_clothes/episode_56.hdf5
/home/zme/data/robot_data/fold_clothes/episode_168.hdf5
/home/zme/data/robot_data/fold_clothes/episode_142.hdf5
/home/zme/data/robot_data/fold_clothes/episode_107.hdf5
/home/zme/data/robot_data/fold_clothes/episode_17.hdf5
/home/zme/data/robot_data/fold_clothes/episode_51.hdf5
/home/zme/data/robot_data/fold_clothes/episode_110.hdf5
/home/zme/data/robot_data/fold_clothes/episode_84.hdf5
/home/zme/data/robot_data/fold_clothes/episode_68.hdf5
/home/zme/data/robot_data/fold_clothes/episode_153.hdf5
/hom